In [3]:
from pathlib import Path
import os

root = Path.cwd()

while not (root / ".git").exists():
    root = root.parent

os.chdir(root)

print("Project root:", root)

Project root: d:\Projects\btc-research-lab


In [4]:
import pandas as pd
import numpy as np

# Load base dataset
df = pd.read_parquet(
    "data/processed/BTCUSDT_5m_regularized.parquet"
)

# Time index
df['open_time'] = pd.to_datetime(df['open_time'])
df = df.set_index('open_time')

print(df.shape)
df.head()

(631296, 5)


,open,high,low,close,volume
open_time,,,,,
2019-01-01 00:00:00,3701.23,3703.72,3695.00,3696.32,85.572181
2019-01-01 00:05:00,3696.30,3697.24,3689.88,3692.34,62.296581
2019-01-01 00:10:00,3692.34,3698.93,3692.34,3697.31,43.105333
2019-01-01 00:15:00,3697.91,3698.75,3693.00,3693.00,48.551084
2019-01-01 00:20:00,3693.44,3695.98,3690.92,3692.18,47.706443


In [5]:
work_df = df.copy()

# H-02 : Compression -> Expansion

work_df['ret'] = work_df['close'].pct_change()

work_df['rolling_vol_48'] = (
    work_df['ret']
    .rolling(48)
    .std()
)

vol_threshold = work_df['rolling_vol_48'].quantile(0.10)

work_df['compression'] = (
    work_df['rolling_vol_48'] <= vol_threshold
)

work_df['fwd_6'] = work_df['close'].shift(-6)/work_df['close'] - 1
work_df['fwd_12'] = work_df['close'].shift(-12)/work_df['close'] - 1
work_df['fwd_24'] = work_df['close'].shift(-24)/work_df['close'] - 1

expansion_threshold = 0.005  # 0.5%

subset = work_df.loc[work_df['compression']].copy()

subset['big_move_12'] = (
    subset['fwd_12'].abs() > expansion_threshold
)

baseline_prob = (
    work_df['fwd_12'].abs() > expansion_threshold
).mean()

event_prob = subset['big_move_12'].mean()

lift = event_prob / baseline_prob

print("="*50)
print("H-02 Compression -> Expansion")
print("="*50)
print("n =", len(subset))
print("baseline_prob =", round(baseline_prob,4))
print("event_prob =", round(event_prob,4))
print("lift =", round(lift,3))

print()
print("mean_abs_fwd6 =", round(subset['fwd_6'].abs().mean(),5))
print("mean_abs_fwd12 =", round(subset['fwd_12'].abs().mean(),5))
print("mean_abs_fwd24 =", round(subset['fwd_24'].abs().mean(),5))

H-02 Compression -> Expansion
n = 63125
baseline_prob = 0.2529
event_prob = 0.0392
lift = 0.155

mean_abs_fwd6 = 0.00104
mean_abs_fwd12 = 0.00149
mean_abs_fwd24 = 0.00216


In [6]:
# ==========================================
# Phase 3.1b
# H-03 : Volatility Expansion Event
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# short-term and medium-term volatility
work_df['vol_12'] = work_df['ret'].rolling(12).std()
work_df['vol_48'] = work_df['ret'].rolling(48).std()

# expansion ratio
work_df['vol_ratio'] = work_df['vol_12'] / work_df['vol_48']

# future returns
work_df['fwd_6'] = work_df['close'].shift(-6) / work_df['close'] - 1
work_df['fwd_12'] = work_df['close'].shift(-12) / work_df['close'] - 1
work_df['fwd_24'] = work_df['close'].shift(-24) / work_df['close'] - 1

# top 5% expansion events
threshold = work_df['vol_ratio'].quantile(0.95)

subset = work_df.loc[work_df['vol_ratio'] >= threshold].copy()

# big move definition
move_threshold = 0.005   # 0.5%

subset['big_move_12'] = subset['fwd_12'].abs() > move_threshold
baseline_big_move = (work_df['fwd_12'].abs() > move_threshold).mean()

event_big_move = subset['big_move_12'].mean()

# direction
long_hit = (subset['fwd_12'] > 0).mean()
short_hit = (subset['fwd_12'] < 0).mean()

print("="*60)
print("H-03 : Volatility Expansion Event")
print("="*60)

print(f"n = {len(subset)}")
print()
print(f"baseline_big_move_prob = {baseline_big_move:.4f}")
print(f"event_big_move_prob    = {event_big_move:.4f}")
print(f"lift                   = {event_big_move / baseline_big_move:.3f}")

print()
print(f"mean_abs_fwd6  = {subset['fwd_6'].abs().mean():.5f}")
print(f"mean_abs_fwd12 = {subset['fwd_12'].abs().mean():.5f}")
print(f"mean_abs_fwd24 = {subset['fwd_24'].abs().mean():.5f}")

print()
print(f"long_hit  = {long_hit:.3f}")
print(f"short_hit = {short_hit:.3f}")

print()
print("mean_fwd6  =", round(subset['fwd_6'].mean(),6))
print("mean_fwd12 =", round(subset['fwd_12'].mean(),6))
print("mean_fwd24 =", round(subset['fwd_24'].mean(),6))

H-03 : Volatility Expansion Event
n = 31560

baseline_big_move_prob = 0.2529
event_big_move_prob    = 0.3283
lift                   = 1.298

mean_abs_fwd6  = 0.00404
mean_abs_fwd12 = 0.00528
mean_abs_fwd24 = 0.00705

long_hit  = 0.522
short_hit = 0.478

mean_fwd6  = 0.00015
mean_fwd12 = 0.000238
mean_fwd24 = 0.00032


In [7]:
# ==========================================
# Phase 3.1c
# H-03b : Volatility Expansion + Momentum Continuation
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# volatility
work_df['vol_12'] = work_df['ret'].rolling(12).std()
work_df['vol_48'] = work_df['ret'].rolling(48).std()

# expansion ratio
work_df['vol_ratio'] = work_df['vol_12'] / work_df['vol_48']

# top 5% expansion events
threshold = work_df['vol_ratio'].quantile(0.95)

# past and future returns
work_df['past_12'] = work_df['close'] / work_df['close'].shift(12) - 1
work_df['fwd_12'] = work_df['close'].shift(-12) / work_df['close'] - 1

# signs
work_df['past_sign'] = np.sign(work_df['past_12'])
work_df['future_sign'] = np.sign(work_df['fwd_12'])

# expansion subset
subset = work_df.loc[work_df['vol_ratio'] >= threshold].copy()

# continuation
subset['continuation'] = (
    subset['past_sign'] == subset['future_sign']
)

# positive momentum
up_subset = subset.loc[subset['past_sign'] > 0]

# negative momentum
down_subset = subset.loc[subset['past_sign'] < 0]

print("="*60)
print("H-03b : Volatility Expansion + Momentum Continuation")
print("="*60)

print("\nALL EVENTS")
print("n =", len(subset))
print("continuation_rate =", round(subset['continuation'].mean(),4))
print("mean_fwd12 =", round(subset['fwd_12'].mean(),6))

print("\nUP MOMENTUM")
print("n =", len(up_subset))
print("continuation_rate =", round(
    (up_subset['future_sign'] > 0).mean(),4))
print("mean_fwd12 =", round(up_subset['fwd_12'].mean(),6))

print("\nDOWN MOMENTUM")
print("n =", len(down_subset))
print("continuation_rate =", round(
    (down_subset['future_sign'] < 0).mean(),4))
print("mean_fwd12 =", round(down_subset['fwd_12'].mean(),6))

H-03b : Volatility Expansion + Momentum Continuation

ALL EVENTS
n = 31560
continuation_rate = 0.4623
mean_fwd12 = 0.000238

UP MOMENTUM
n = 14737
continuation_rate = 0.4831
mean_fwd12 = 3.4e-05

DOWN MOMENTUM
n = 16821
continuation_rate = 0.4441
mean_fwd12 = 0.000417


In [8]:
# ==========================================
# Phase 3.2
# H-04 : 3-Bar Pattern Mining
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# future return
work_df['fwd_6'] = work_df['close'].shift(-6) / work_df['close'] - 1

# sign encoding
work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

# 3-bar patterns
work_df['pattern'] = (
    work_df['sign'].shift(2) +
    work_df['sign'].shift(1) +
    work_df['sign']
)

patterns = sorted(work_df['pattern'].dropna().unique())

print("="*70)
print("H-04 : 3-Bar Pattern Mining")
print("="*70)

results = []

for p in patterns:

    subset = work_df.loc[work_df['pattern'] == p]

    n = len(subset)

    mean_fwd6 = subset['fwd_6'].mean()
    median_fwd6 = subset['fwd_6'].median()
    hit_rate = (subset['fwd_6'] > 0).mean()

    results.append([
        p,
        n,
        mean_fwd6,
        median_fwd6,
        hit_rate
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        'pattern',
        'n',
        'mean_fwd6',
        'median_fwd6',
        'hit_rate'
    ]
)

results_df = results_df.sort_values(
    by='mean_fwd6',
    ascending=False
)

print(results_df.round(5))

H-04 : 3-Bar Pattern Mining
  pattern      n  mean_fwd6  median_fwd6  hit_rate
0     DDD  70691    0.00018      0.00027   0.54540
4     UDD  81211    0.00016      0.00020   0.53341
2     DUD  82961    0.00012      0.00014   0.52401
6     UUD  81123    0.00004      0.00001   0.50110
1     DDU  81212    0.00003      0.00005   0.50841
5     UDU  82872   -0.00002     -0.00008   0.48636
3     DUU  81123   -0.00008     -0.00011   0.48121
7     UUU  70101   -0.00008     -0.00023   0.46106


In [9]:
# ==========================================
# Phase 3.2b
# H-04b : Horizon Analysis for DDD and UUU
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# sign encoding
work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

# pattern
work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

# future horizons
horizons = [3, 6, 12, 24]

for h in horizons:
    work_df[f'fwd_{h}'] = (
        work_df['close'].shift(-h) / work_df['close'] - 1
    )

patterns_to_test = ['DDD', 'UUU']

print("=" * 80)
print("H-04b : Horizon Analysis")
print("=" * 80)

for p in patterns_to_test:

    subset = work_df[work_df['pattern'] == p]

    print(f"\nPattern: {p}")
    print("-" * 50)

    for h in horizons:

        mean_ret = subset[f'fwd_{h}'].mean()
        median_ret = subset[f'fwd_{h}'].median()
        hit_rate = (subset[f'fwd_{h}'] > 0).mean()

        print(
            f"FWD{h:2d} | "
            f"mean={mean_ret:.5f} | "
            f"median={median_ret:.5f} | "
            f"hit={hit_rate:.3f}"
        )

H-04b : Horizon Analysis

Pattern: DDD
--------------------------------------------------
FWD 3 | mean=0.00015 | median=0.00022 | hit=0.551
FWD 6 | mean=0.00018 | median=0.00027 | hit=0.545
FWD12 | mean=0.00027 | median=0.00037 | hit=0.546
FWD24 | mean=0.00033 | median=0.00047 | hit=0.542

Pattern: UUU
--------------------------------------------------
FWD 3 | mean=-0.00007 | median=-0.00020 | hit=0.455
FWD 6 | mean=-0.00008 | median=-0.00023 | hit=0.461
FWD12 | mean=0.00002 | median=-0.00022 | hit=0.473
FWD24 | mean=0.00016 | median=-0.00021 | hit=0.481


In [10]:
# ==========================================
# Phase 3.2c
# H-04c : DDD Magnitude Segmentation
# ==========================================

work_df = df.copy()

work_df['ret'] = work_df['close'].pct_change()

# pattern
work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

# cumulative return of three bars
work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

# future returns
work_df['fwd_6'] = work_df['close'].shift(-6)/work_df['close'] - 1
work_df['fwd_12'] = work_df['close'].shift(-12)/work_df['close'] - 1
work_df['fwd_24'] = work_df['close'].shift(-24)/work_df['close'] - 1

subset = work_df[work_df['pattern'] == 'DDD'].copy()

# divide into quartiles by strength
subset['group'] = pd.qcut(
    subset['ddd_strength'],
    q=4,
    labels=['Q1_strongest',
            'Q2',
            'Q3',
            'Q4_weakest']
)

print("="*80)
print("DDD Magnitude Segmentation")
print("="*80)

for g in subset['group'].unique():

    s = subset[subset['group']==g]

    print(f"\n{g}")
    print("n =", len(s))
    print("mean_fwd6  =", round(s['fwd_6'].mean(),6))
    print("mean_fwd12 =", round(s['fwd_12'].mean(),6))
    print("mean_fwd24 =", round(s['fwd_24'].mean(),6))
    print("hit_rate24 =", round((s['fwd_24']>0).mean(),3))

DDD Magnitude Segmentation

Q2
n = 17673
mean_fwd6  = 7.4e-05
mean_fwd12 = 0.000147
mean_fwd24 = 0.00014
hit_rate24 = 0.548

Q3
n = 17672
mean_fwd6  = 0.000155
mean_fwd12 = 0.000195
mean_fwd24 = 0.00014
hit_rate24 = 0.539

Q4_weakest
n = 17673
mean_fwd6  = 0.000138
mean_fwd12 = 0.000194
mean_fwd24 = 0.000277
hit_rate24 = 0.524

Q1_strongest
n = 17673
mean_fwd6  = 0.000343
mean_fwd12 = 0.000546
mean_fwd24 = 0.000743
hit_rate24 = 0.558


In [11]:
# ==========================================
# Phase 3.2d
# Overlap between H-04c and H-01
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# pattern
work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

# DDD strength
work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

# strongest quartile
ddd_subset = work_df[work_df['pattern']=='DDD'].copy()

threshold_q1 = ddd_subset['ddd_strength'].quantile(0.25)

work_df['H04_Q1'] = (
    (work_df['pattern']=='DDD')
    &
    (work_df['ddd_strength'] <= threshold_q1)
)

# H01 definition
shock_threshold = work_df['ret'].quantile(0.01)

work_df['H01'] = (
    work_df['ret'] <= shock_threshold
)

# counts
n_h04 = work_df['H04_Q1'].sum()
n_h01 = work_df['H01'].sum()
n_overlap = (work_df['H04_Q1'] & work_df['H01']).sum()

print("="*70)
print("Overlap Analysis")
print("="*70)

print("H04_Q1 =", n_h04)
print("H01 =", n_h01)
print("Overlap =", n_overlap)

print()
print("Overlap / H04_Q1 =", round(n_overlap/n_h04,3))
print("Overlap / H01 =", round(n_overlap/n_h01,3))

Overlap Analysis
H04_Q1 = 17673
H01 = 6313
Overlap = 2016

Overlap / H04_Q1 = 0.114
Overlap / H01 = 0.319


In [12]:
# ==========================================
# Phase 3.2e
# DDD-Q1 across volatility regimes
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# volatility regime
work_df['rolling_vol_48'] = (
    work_df['ret']
    .rolling(48)
    .std()
)

work_df['vol_regime'] = pd.qcut(
    work_df['rolling_vol_48'],
    q=3,
    labels=['Low','Mid','High']
)

# pattern
work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

# DDD strength
work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

ddd = work_df[work_df['pattern']=='DDD'].copy()

threshold_q1 = ddd['ddd_strength'].quantile(0.25)

work_df['DDD_Q1'] = (
    (work_df['pattern']=='DDD')
    &
    (work_df['ddd_strength'] <= threshold_q1)
)

# future returns
work_df['fwd_24'] = (
    work_df['close'].shift(-24) /
    work_df['close'] - 1
)

subset = work_df[work_df['DDD_Q1']]

print("="*70)
print("DDD_Q1 by Volatility Regime")
print("="*70)

for regime in ['Low','Mid','High']:

    s = subset[subset['vol_regime']==regime]

    print(f"\n{regime}")
    print("n =", len(s))
    print("mean_fwd24 =", round(s['fwd_24'].mean(),6))
    print("median_fwd24 =", round(s['fwd_24'].median(),6))
    print("hit_rate =", round((s['fwd_24']>0).mean(),3))

DDD_Q1 by Volatility Regime

Low
n = 688
mean_fwd24 = 0.000361
median_fwd24 = 0.001037
hit_rate = 0.576

Mid
n = 4544
mean_fwd24 = -5.8e-05
median_fwd24 = 0.000832
hit_rate = 0.551

High
n = 12441
mean_fwd24 = 0.001056
median_fwd24 = 0.001527
hit_rate = 0.559


In [13]:
# ==========================================
# Phase 3.3
# H04c First Backtest
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# pattern
work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

# DDD strength
work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

ddd = work_df[work_df['pattern'] == 'DDD']

threshold_q1 = ddd['ddd_strength'].quantile(0.25)

work_df['signal'] = (
    (work_df['pattern'] == 'DDD')
    &
    (work_df['ddd_strength'] <= threshold_q1)
)

# trade return
work_df['trade_ret'] = (
    work_df['close'].shift(-24) /
    work_df['close']
    - 1
)

signal_idx = work_df.index[work_df['signal']]

trade_returns = []

last_exit = work_df.index[0]

for idx in signal_idx:

    if idx <= last_exit:
        continue

    loc = work_df.index.get_loc(idx)

    if loc + 24 >= len(work_df):
        break

    trade_returns.append(
        work_df.iloc[loc]['trade_ret']
    )

    last_exit = work_df.index[loc + 24]

trade_returns = pd.Series(trade_returns)

print("="*70)
print("H04c Backtest")
print("="*70)

print("n_trades =", len(trade_returns))
print("mean =", round(trade_returns.mean(),6))
print("median =", round(trade_returns.median(),6))
print("hit_rate =", round((trade_returns>0).mean(),3))
print("profit_factor =",
      round(
          trade_returns[trade_returns>0].sum()
          /
          abs(trade_returns[trade_returns<0].sum()),
          3
      ))
print("sharpe =",
      round(
          np.sqrt(365*24*12)
          * trade_returns.mean()
          / trade_returns.std(),
          3
      ))

H04c Backtest
n_trades = 7013
mean = 0.000482
median = 0.000934
hit_rate = 0.549
profit_factor = 1.119
sharpe = 11.761


In [14]:
# ==========================================
# Phase 3.3b
# H04c Cost Sensitivity
# ==========================================

work_df = df.copy()

work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

ddd = work_df[work_df['pattern'] == 'DDD']

threshold_q1 = ddd['ddd_strength'].quantile(0.25)

work_df['signal'] = (
    (work_df['pattern'] == 'DDD')
    &
    (work_df['ddd_strength'] <= threshold_q1)
)

work_df['trade_ret'] = (
    work_df['close'].shift(-24)
    / work_df['close']
    - 1
)

signal_idx = work_df.index[work_df['signal']]

trade_returns = []

last_exit = work_df.index[0]

for idx in signal_idx:

    if idx <= last_exit:
        continue

    loc = work_df.index.get_loc(idx)

    if loc + 24 >= len(work_df):
        break

    trade_returns.append(
        work_df.iloc[loc]['trade_ret']
    )

    last_exit = work_df.index[loc + 24]

trade_returns = pd.Series(trade_returns)

print("="*70)
print("H04c Cost Sensitivity")
print("="*70)

costs_bp = [0,2,4,6,8,10]

for bp in costs_bp:

    cost = bp / 10000

    net_ret = trade_returns - cost

    pf = (
        net_ret[net_ret > 0].sum()
        / abs(net_ret[net_ret < 0].sum())
    )

    print()
    print(f"Cost = {bp} bp")
    print("mean =", round(net_ret.mean(),6))
    print("median =", round(net_ret.median(),6))
    print("hit_rate =", round((net_ret > 0).mean(),3))
    print("profit_factor =", round(pf,3))

H04c Cost Sensitivity

Cost = 0 bp
mean = 0.000482
median = 0.000934
hit_rate = 0.549
profit_factor = 1.119

Cost = 2 bp
mean = 0.000282
median = 0.000734
hit_rate = 0.541
profit_factor = 1.068

Cost = 4 bp
mean = 8.2e-05
median = 0.000534
hit_rate = 0.529
profit_factor = 1.019

Cost = 6 bp
mean = -0.000118
median = 0.000334
hit_rate = 0.518
profit_factor = 0.973

Cost = 8 bp
mean = -0.000318
median = 0.000134
hit_rate = 0.507
profit_factor = 0.928

Cost = 10 bp
mean = -0.000518
median = -6.6e-05
hit_rate = 0.496
profit_factor = 0.885


In [15]:
# ==========================================
# Phase 3.3c
# H04c Exit Horizon Analysis
# ==========================================

work_df = df.copy()

work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

ddd = work_df[work_df['pattern'] == 'DDD']

threshold_q1 = ddd['ddd_strength'].quantile(0.25)

work_df['signal'] = (
    (work_df['pattern'] == 'DDD')
    &
    (work_df['ddd_strength'] <= threshold_q1)
)

horizons = [6,12,24,36,48,72]

signal_idx = work_df.index[work_df['signal']]

print("="*70)
print("H04c Exit Horizon Analysis")
print("="*70)

for h in horizons:

    work_df['trade_ret'] = (
        work_df['close'].shift(-h)
        / work_df['close']
        - 1
    )

    trade_returns = []

    last_exit = work_df.index[0]

    for idx in signal_idx:

        if idx <= last_exit:
            continue

        loc = work_df.index.get_loc(idx)

        if loc + h >= len(work_df):
            break

        trade_returns.append(
            work_df.iloc[loc]['trade_ret']
        )

        last_exit = work_df.index[loc+h]

    trade_returns = pd.Series(trade_returns)

    pf = (
        trade_returns[trade_returns > 0].sum()
        /
        abs(trade_returns[trade_returns < 0].sum())
    )

    print()
    print(f"Horizon {h}")
    print("n =", len(trade_returns))
    print("mean =", round(trade_returns.mean(),6))
    print("hit_rate =", round((trade_returns>0).mean(),3))
    print("profit_factor =", round(pf,3))

H04c Exit Horizon Analysis

Horizon 6
n = 10541
mean = 0.000338
hit_rate = 0.565
profit_factor = 1.144

Horizon 12
n = 8926
mean = 0.000377
hit_rate = 0.559
profit_factor = 1.12

Horizon 24
n = 7013
mean = 0.000482
hit_rate = 0.549
profit_factor = 1.119

Horizon 36
n = 5879
mean = 0.000603
hit_rate = 0.556
profit_factor = 1.128

Horizon 48
n = 5082
mean = 0.000524
hit_rate = 0.547
profit_factor = 1.096

Horizon 72
n = 4092
mean = 0.000731
hit_rate = 0.544
profit_factor = 1.113


In [16]:
# ==========================================
# Phase 3.4
# H04c Out-of-Sample Test
# ==========================================

work_df = df.copy()

work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

# ---------- split ----------
train = work_df.loc[:'2022-12-31'].copy()
test = work_df.loc['2023-01-01':].copy()

# learn threshold only on train
train_ddd = train[train['pattern']=='DDD']

threshold_q1 = train_ddd['ddd_strength'].quantile(0.25)

# apply to both
train['signal'] = (
    (train['pattern']=='DDD')
    &
    (train['ddd_strength'] <= threshold_q1)
)

test['signal'] = (
    (test['pattern']=='DDD')
    &
    (test['ddd_strength'] <= threshold_q1)
)

# horizon
H = 36

for d in [train,test]:

    d['trade_ret'] = (
        d['close'].shift(-H)
        /
        d['close']
        - 1
    )

def backtest(dataset):

    signal_idx = dataset.index[dataset['signal']]

    trade_returns = []

    last_exit = dataset.index[0]

    for idx in signal_idx:

        if idx <= last_exit:
            continue

        loc = dataset.index.get_loc(idx)

        if loc + H >= len(dataset):
            break

        trade_returns.append(
            dataset.iloc[loc]['trade_ret']
        )

        last_exit = dataset.index[loc+H]

    tr = pd.Series(trade_returns)

    pf = (
        tr[tr>0].sum()
        /
        abs(tr[tr<0].sum())
    )

    return len(tr),tr.mean(),(tr>0).mean(),pf

print("="*70)
print("H04c OOS")
print("="*70)

for name,d in [('TRAIN',train),('TEST',test)]:

    n,mean_,hit,pf = backtest(d)

    print()
    print(name)
    print("n =",n)
    print("mean =",round(mean_,6))
    print("hit =",round(hit,3))
    print("pf =",round(pf,3))

H04c OOS

TRAIN
n = 3872
mean = 0.000531
hit = 0.552
pf = 1.101

TEST
n = 1199
mean = 0.000639
hit = 0.558
pf = 1.159


In [17]:
# ==========================================
# Phase 3.5a
# Parameter Stability
# ==========================================

work_df = df.copy()

work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
      work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

work_df['trade_ret'] = (
    work_df['close'].shift(-36)
    / work_df['close']
    - 1
)

ddd = work_df[work_df['pattern']=="DDD"]

quantiles = [0.10,0.20,0.25,0.30,0.40]

print("="*70)
print("Parameter Stability")
print("="*70)

for q in quantiles:

    th = ddd['ddd_strength'].quantile(q)

    subset = work_df[
        (work_df['pattern']=="DDD")
        &
        (work_df['ddd_strength']<=th)
    ]

    tr = subset['trade_ret'].dropna()

    pf = (
        tr[tr>0].sum()
        /
        abs(tr[tr<0].sum())
    )

    print()
    print("Q =",q)
    print("n =",len(tr))
    print("mean =",round(tr.mean(),6))
    print("hit =",round((tr>0).mean(),3))
    print("pf =",round(pf,3))

Parameter Stability

Q = 0.1
n = 7070
mean = 0.001381
hit = 0.564
pf = 1.196

Q = 0.2
n = 14139
mean = 0.000729
hit = 0.557
pf = 1.12

Q = 0.25
n = 17673
mean = 0.000647
hit = 0.556
pf = 1.112

Q = 0.3
n = 21207
mean = 0.000521
hit = 0.553
pf = 1.094

Q = 0.4
n = 28276
mean = 0.000471
hit = 0.549
pf = 1.092


In [18]:
# ==========================================
# Phase 3.5b
# Year-by-Year Robustness
# ==========================================

work_df = df.copy()

work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

# Q10 threshold
ddd = work_df[work_df['pattern'] == 'DDD']
threshold = ddd['ddd_strength'].quantile(0.10)

work_df['signal'] = (
    (work_df['pattern'] == 'DDD')
    &
    (work_df['ddd_strength'] <= threshold)
)

# horizon = 36
work_df['trade_ret'] = (
    work_df['close'].shift(-36)
    / work_df['close']
    - 1
)

work_df['year'] = work_df.index.year

print("="*70)
print("Year-by-Year Robustness")
print("="*70)

for year in sorted(work_df['year'].unique()):

    subset = work_df[
        (work_df['year'] == year)
        &
        (work_df['signal'])
    ]

    tr = subset['trade_ret'].dropna()

    if len(tr) == 0:
        continue

    pf = (
        tr[tr > 0].sum()
        /
        abs(tr[tr < 0].sum())
    )

    print()
    print(year)
    print("n =", len(tr))
    print("mean =", round(tr.mean(),6))
    print("hit =", round((tr > 0).mean(),3))
    print("pf =", round(pf,3))

Year-by-Year Robustness

2019
n = 980
mean = 0.002584
hit = 0.581
pf = 1.432

2020
n = 965
mean = 0.001149
hit = 0.613
pf = 1.132

2021
n = 2556
mean = 0.002142
hit = 0.563
pf = 1.28

2022
n = 1330
mean = -0.001136
hit = 0.509
pf = 0.85

2023
n = 394
mean = 0.002779
hit = 0.614
pf = 1.881

2024
n = 845
mean = 0.001261
hit = 0.554
pf = 1.225


In [19]:
# ==========================================
# Phase 3.5c
# Remove Best Trades Robustness
# ==========================================

work_df = df.copy()

work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

ddd = work_df[work_df['pattern']=='DDD']

threshold = ddd['ddd_strength'].quantile(0.10)

subset = work_df[
    (work_df['pattern']=='DDD')
    &
    (work_df['ddd_strength']<=threshold)
].copy()

subset['trade_ret'] = (
    subset['close'].shift(-36)
    /
    subset['close']
    - 1
)

tr = subset['trade_ret'].dropna()

print("="*70)
print("Tail Dependence")
print("="*70)

for pct in [0,1,2,5,10]:

    cutoff = np.percentile(tr,100-pct) if pct>0 else np.inf

    filtered = tr[tr<=cutoff]

    pf = (
        filtered[filtered>0].sum()
        /
        abs(filtered[filtered<0].sum())
    )

    print()
    print(f"Remove top {pct}%")
    print("n =",len(filtered))
    print("mean =",round(filtered.mean(),6))
    print("hit =",round((filtered>0).mean(),3))
    print("pf =",round(pf,3))

Tail Dependence

Remove top 0%
n = 7034
mean = 0.02419
hit = 0.531
pf = 1.655

Remove top 1%
n = 6963
mean = 0.020054
hit = 0.526
pf = 1.537

Remove top 2%
n = 6893
mean = 0.016709
hit = 0.522
pf = 1.443

Remove top 5%
n = 6682
mean = 0.008214
hit = 0.507
pf = 1.211

Remove top 10%
n = 6330
mean = -0.003551
hit = 0.479
pf = 0.914


In [20]:
# ==========================================
# Phase 3.5d
# H04c Macro Regime Analysis
# ==========================================

work_df = df.copy()

# returns
work_df['ret'] = work_df['close'].pct_change()

# sign pattern
work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

# DDD strength
work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

# Q10 threshold
ddd = work_df[work_df['pattern'] == 'DDD']
threshold = ddd['ddd_strength'].quantile(0.10)

work_df['signal'] = (
    (work_df['pattern'] == 'DDD')
    &
    (work_df['ddd_strength'] <= threshold)
)

# 36-bar future return (correct)
H = 36
work_df['trade_ret'] = (
    work_df['close'].shift(-H)
    / work_df['close']
    - 1
)

# daily MA200 regime
daily_close = work_df['close'].resample('1D').last()
daily_ma200 = daily_close.rolling(200).mean()

daily_regime = (
    daily_close > daily_ma200
).rename('bull')

work_df = work_df.join(
    daily_regime,
    on=work_df.index.floor('D')
)

subset = work_df[work_df['signal']].copy()

print("="*70)
print("H04c Macro Regime Analysis")
print("="*70)

for regime_name, mask in [
    ("Bull", subset['bull']==True),
    ("Bear", subset['bull']==False)
]:

    tr = subset.loc[mask, 'trade_ret'].dropna()

    pf = (
        tr[tr>0].sum()
        /
        abs(tr[tr<0].sum())
    )

    print()
    print(regime_name)
    print("n =", len(tr))
    print("mean =", round(tr.mean(),6))
    print("hit =", round((tr>0).mean(),3))
    print("pf =", round(pf,3))

H04c Macro Regime Analysis

Bull
n = 3466
mean = 0.002264
hit = 0.583
pf = 1.383

Bear
n = 3604
mean = 0.000532
hit = 0.545
pf = 1.065


In [21]:
# ============================================================
# Phase 3.6
# Event Backtester for H04c
# ============================================================

work_df = df.copy()

# ---------- signal construction ----------
work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
      work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

# Q10 threshold
ddd = work_df[work_df['pattern'] == 'DDD']
threshold = ddd['ddd_strength'].quantile(0.10)

work_df['signal'] = (
    (work_df['pattern'] == 'DDD')
    &
    (work_df['ddd_strength'] <= threshold)
)

# ---------- parameters ----------
HOLD_BARS = 36
COST_BP = 4          # round-trip cost
cost = COST_BP / 10000

# ---------- event backtest ----------
trade_list = []

signal_idx = work_df.index[work_df['signal']]

last_exit_loc = -1

for idx in signal_idx:

    entry_loc = work_df.index.get_loc(idx)

    # no overlap
    if entry_loc <= last_exit_loc:
        continue

    exit_loc = entry_loc + HOLD_BARS

    if exit_loc >= len(work_df):
        break

    entry_time = work_df.index[entry_loc]
    exit_time = work_df.index[exit_loc]

    entry_price = work_df.iloc[entry_loc]['close']
    exit_price = work_df.iloc[exit_loc]['close']

    gross_ret = exit_price / entry_price - 1
    net_ret = gross_ret - cost

    trade_list.append([
        entry_time,
        exit_time,
        entry_price,
        exit_price,
        gross_ret,
        net_ret,
        HOLD_BARS
    ])

    last_exit_loc = exit_loc

# ---------- trade dataframe ----------
trade_df = pd.DataFrame(
    trade_list,
    columns=[
        'entry_time',
        'exit_time',
        'entry_price',
        'exit_price',
        'gross_ret',
        'net_ret',
        'duration'
    ]
)

trade_df['year'] = trade_df['entry_time'].dt.year

# ---------- equity ----------
trade_df['equity'] = (1 + trade_df['net_ret']).cumprod()

rolling_max = trade_df['equity'].cummax()
drawdown = trade_df['equity'] / rolling_max - 1

# ---------- statistics ----------
wins = trade_df.loc[trade_df['net_ret'] > 0, 'net_ret']
losses = trade_df.loc[trade_df['net_ret'] <= 0, 'net_ret']

profit_factor = wins.sum() / abs(losses.sum())

sharpe = (
    np.sqrt(len(trade_df))
    * trade_df['net_ret'].mean()
    / trade_df['net_ret'].std()
)

print("="*70)
print("Phase 3.6 Event Backtester")
print("="*70)

print("Trades        :", len(trade_df))
print("Mean net ret  :", round(trade_df['net_ret'].mean(),6))
print("Median net ret:", round(trade_df['net_ret'].median(),6))
print("Hit rate      :", round((trade_df['net_ret']>0).mean(),3))
print("Profit factor :", round(profit_factor,3))
print("Sharpe        :", round(sharpe,3))
print("Max drawdown  :", round(drawdown.min(),3))
print("Final equity  :", round(trade_df['equity'].iloc[-1],3))

print("\nFirst trades:")
print(trade_df.head())

print("\nLast trades:")
print(trade_df.tail())

Phase 3.6 Event Backtester
Trades        : 2840
Mean net ret  : 0.000579
Median net ret: 0.001343
Hit rate      : 0.549
Profit factor : 1.097
Sharpe        : 1.627
Max drawdown  : -0.513
Final equity  : 3.067

First trades:
           entry_time           exit_time  entry_price  exit_price  gross_ret  \
0 2019-01-04 14:40:00 2019-01-04 17:40:00      3712.36     3738.72   0.007101   
1 2019-01-07 09:15:00 2019-01-07 12:15:00      3966.05     3974.00   0.002005   
2 2019-01-08 15:55:00 2019-01-08 18:55:00      4000.48     3968.46  -0.008004   
3 2019-01-10 06:30:00 2019-01-10 09:30:00      3812.86     3743.00  -0.018322   
4 2019-01-10 16:10:00 2019-01-10 19:10:00      3543.99     3563.87   0.005609   

    net_ret  duration  year    equity  
0  0.006701        36  2019  1.006701  
1  0.001605        36  2019  1.008316  
2 -0.008404        36  2019  0.999842  
3 -0.018722        36  2019  0.981123  
4  0.005209        36  2019  0.986234  

Last trades:
              entry_time           

In [22]:
# ============================================================
# Phase 3.6b
# Trade Performance by Year
# ============================================================

print("="*70)
print("Performance by Year")
print("="*70)

for year in sorted(trade_df['year'].unique()):

    tr = trade_df.loc[
        trade_df['year']==year,
        'net_ret'
    ]

    pf = (
        tr[tr>0].sum()
        /
        abs(tr[tr<=0].sum())
    )

    equity = (1+tr).prod()

    print()
    print(year)
    print("n =",len(tr))
    print("mean =",round(tr.mean(),6))
    print("hit =",round((tr>0).mean(),3))
    print("pf =",round(pf,3))
    print("year_return =",round(equity-1,3))

Performance by Year

2019
n = 428
mean = 0.000865
hit = 0.549
pf = 1.149
year_return = 0.34

2020
n = 420
mean = 0.001437
hit = 0.588
pf = 1.249
year_return = 0.592

2021
n = 886
mean = 0.000626
hit = 0.543
pf = 1.095
year_return = 0.477

2022
n = 528
mean = -0.00063
hit = 0.511
pf = 0.901
year_return = -0.337

2023
n = 212
mean = 0.001658
hit = 0.604
pf = 1.451
year_return = 0.396

2024
n = 366
mean = 0.000262
hit = 0.544
pf = 1.047
year_return = 0.052


In [23]:
# ============================================================
# Phase 3.6c
# Performance by Volatility Regime
# ============================================================

work_df = df.copy()

# ---------- volatility regime ----------
work_df['ret'] = work_df['close'].pct_change()

work_df['rv'] = (
    work_df['ret']
    .rolling(48)
    .std()
)

work_df['rv_ewm'] = work_df['rv'].ewm(span=10).mean()

work_df['vol_regime'] = pd.qcut(
    work_df['rv_ewm'],
    q=3,
    labels=['Low','Mid','High']
)

# attach regime to trades
trade_df['vol_regime'] = (
    work_df.loc[
        trade_df['entry_time'],
        'vol_regime'
    ].values
)

print("="*70)
print("Performance by Volatility Regime")
print("="*70)

for regime in ['Low','Mid','High']:

    tr = trade_df.loc[
        trade_df['vol_regime']==regime,
        'net_ret'
    ]

    pf = (
        tr[tr>0].sum()
        /
        abs(tr[tr<=0].sum())
    )

    eq = (1+tr).prod()

    print()
    print(regime)
    print("n =",len(tr))
    print("mean =",round(tr.mean(),6))
    print("hit =",round((tr>0).mean(),3))
    print("pf =",round(pf,3))
    print("equity =",round(eq,3))

Performance by Volatility Regime

Low
n = 163
mean = -0.000403
hit = 0.509
pf = 0.894
equity = 0.929

Mid
n = 826
mean = -0.000222
hit = 0.547
pf = 0.954
equity = 0.775

High
n = 1851
mean = 0.001022
hit = 0.554
pf = 1.154
equity = 4.262


In [24]:
# ============================================================
# Phase 3.6d
# High Vol Only Backtest
# ============================================================

hv_trades = trade_df[
    trade_df['vol_regime'] == 'High'
].copy()

hv_trades['equity'] = (
    1 + hv_trades['net_ret']
).cumprod()

rolling_max = hv_trades['equity'].cummax()

dd = hv_trades['equity'] / rolling_max - 1

tr = hv_trades['net_ret']

pf = (
    tr[tr>0].sum()
    /
    abs(tr[tr<=0].sum())
)

sharpe = (
    np.sqrt(len(tr))
    * tr.mean()
    / tr.std()
)

print("="*70)
print("High Vol Only Backtest")
print("="*70)

print("n =",len(tr))
print("mean =",round(tr.mean(),6))
print("median =",round(tr.median(),6))
print("hit =",round((tr>0).mean(),3))
print("pf =",round(pf,3))
print("sharpe =",round(sharpe,3))
print("max_dd =",round(dd.min(),3))
print("final_equity =",round(hv_trades['equity'].iloc[-1],3))

High Vol Only Backtest
n = 1851
mean = 0.001022
median = 0.001955
hit = 0.554
pf = 1.154
sharpe = 2.04
max_dd = -0.478
final_equity = 4.262


In [25]:
# ============================================================
# Phase 3.6e
# High Vol + Bull Filter
# ============================================================

# daily bull regime
daily_close = df['close'].resample('1D').last()

daily_ma200 = daily_close.rolling(200).mean()

bull_regime = (daily_close > daily_ma200).rename('bull')

work_df = work_df.join(
    bull_regime,
    on=work_df.index.floor('D')
)

# attach bull flag to trades
trade_df['bull'] = (
    work_df.loc[
        trade_df['entry_time'],
        'bull'
    ].values
)

subset = trade_df[
    (trade_df['vol_regime']=='High')
    &
    (trade_df['bull']==True)
].copy()

subset['equity'] = (1+subset['net_ret']).cumprod()

rolling_max = subset['equity'].cummax()

dd = subset['equity']/rolling_max - 1

tr = subset['net_ret']

pf = (
    tr[tr>0].sum()
    /
    abs(tr[tr<=0].sum())
)

sharpe = (
    np.sqrt(len(tr))
    * tr.mean()
    / tr.std()
)

print("="*70)
print("High Vol + Bull Filter")
print("="*70)

print("n =",len(tr))
print("mean =",round(tr.mean(),6))
print("median =",round(tr.median(),6))
print("hit =",round((tr>0).mean(),3))
print("pf =",round(pf,3))
print("sharpe =",round(sharpe,3))
print("max_dd =",round(dd.min(),3))
print("final_equity =",round(subset['equity'].iloc[-1],3))

High Vol + Bull Filter
n = 901
mean = 0.001388
median = 0.002305
hit = 0.578
pf = 1.239
sharpe = 2.292
max_dd = -0.245
final_equity = 3.008


In [26]:
# ======================================================
# Compare Filters
# ======================================================

systems = {
    "H04c" :
        trade_df.index==trade_df.index,

    "HighVol" :
        trade_df['vol_regime']=="High",

    "Bull" :
        trade_df['bull']==True,

    "HighVol+Bull" :
        (trade_df['vol_regime']=="High")
        &
        (trade_df['bull']==True)
}

print("="*70)
print("Filter Comparison")
print("="*70)

for name,mask in systems.items():

    tr = trade_df.loc[mask,'net_ret']

    equity = (1+tr).cumprod()

    dd = equity/equity.cummax()-1

    pf = tr[tr>0].sum()/abs(tr[tr<=0].sum())

    sharpe = np.sqrt(len(tr))*tr.mean()/tr.std()

    print()
    print(name)
    print("n =",len(tr))
    print("mean =",round(tr.mean(),6))
    print("hit =",round((tr>0).mean(),3))
    print("pf =",round(pf,3))
    print("sharpe =",round(sharpe,3))
    print("maxdd =",round(dd.min(),3))
    print("equity =",round(equity.iloc[-1],3))

Filter Comparison

H04c
n = 2840
mean = 0.000579
hit = 0.549
pf = 1.097
sharpe = 1.627
maxdd = -0.513
equity = 3.067

HighVol
n = 1851
mean = 0.001022
hit = 0.554
pf = 1.154
sharpe = 2.04
maxdd = -0.478
equity = 4.262

Bull
n = 1435
mean = 0.000787
hit = 0.565
pf = 1.148
sharpe = 1.836
maxdd = -0.285
equity = 2.559

HighVol+Bull
n = 901
mean = 0.001388
hit = 0.578
pf = 1.239
sharpe = 2.292
maxdd = -0.245
equity = 3.008


In [27]:
# ============================================================
# Phase 3.7a
# Exposure Analysis
# ============================================================

bars_in_trade = len(hv_trades) * HOLD_BARS

total_bars = len(df)

exposure = bars_in_trade / total_bars

years = (
    (df.index[-1] - df.index[0]).days
    / 365.25
)

cagr = (
    hv_trades['equity'].iloc[-1]
    ** (1 / years)
    - 1
)

print("="*70)
print("Exposure Analysis")
print("="*70)

print("Years =", round(years,2))
print("Final equity =", round(hv_trades['equity'].iloc[-1],3))
print("CAGR =", round(cagr,3))
print("Exposure =", round(exposure,3))
print("Exposure-adjusted CAGR =", round(cagr/exposure,3))

Exposure Analysis
Years = 6.0
Final equity = 4.262
CAGR = 0.273
Exposure = 0.106
Exposure-adjusted CAGR = 2.59


In [28]:
# ============================================================
# Phase 3.7b
# MAE / MFE Analysis
# ============================================================

trade_df['mae'] = np.nan
trade_df['mfe'] = np.nan

for i,row in trade_df.iterrows():

    entry_loc = df.index.get_loc(row['entry_time'])
    exit_loc = df.index.get_loc(row['exit_time'])

    path = df.iloc[entry_loc:exit_loc+1]

    entry_price = row['entry_price']

    min_price = path['low'].min()
    max_price = path['high'].max()

    mae = min_price/entry_price - 1
    mfe = max_price/entry_price - 1

    trade_df.loc[i,'mae'] = mae
    trade_df.loc[i,'mfe'] = mfe

wins = trade_df[trade_df['net_ret']>0]
losses = trade_df[trade_df['net_ret']<=0]

print("="*70)
print("MAE / MFE Analysis")
print("="*70)

print("\nWinning trades")
print("median MAE =", round(wins['mae'].median(),4))
print("median MFE =", round(wins['mfe'].median(),4))

print("\nLosing trades")
print("median MAE =", round(losses['mae'].median(),4))
print("median MFE =", round(losses['mfe'].median(),4))

print("\n95th percentile MAE (wins) =", round(wins['mae'].quantile(0.05),4))
print("95th percentile MAE (all)  =", round(trade_df['mae'].quantile(0.05),4))

MAE / MFE Analysis

Winning trades
median MAE = -0.0055
median MFE = 0.015

Losing trades
median MAE = -0.0178
median MFE = 0.007

95th percentile MAE (wins) = -0.021
95th percentile MAE (all)  = -0.0457


In [29]:
# ============================================================
# Phase 3.7c
# Stop Loss Sweep
# ============================================================

stop_levels = [0.01,0.015,0.02,0.025,0.03,0.04]

print("="*80)
print("Stop Loss Sweep")
print("="*80)

for sl in stop_levels:

    rets = []

    for _,row in trade_df.iterrows():

        entry_loc = df.index.get_loc(row['entry_time'])
        exit_loc = df.index.get_loc(row['exit_time'])

        path = df.iloc[entry_loc:exit_loc+1]

        entry_price = row['entry_price']

        stop_price = entry_price*(1-sl)

        stop_hit = (path['low'] <= stop_price)

        if stop_hit.any():

            stop_bar = stop_hit.idxmax()

            exit_price = stop_price

        else:

            exit_price = row['exit_price']

        ret = exit_price/entry_price - 1 - 0.0004

        rets.append(ret)

    tr = pd.Series(rets)

    pf = tr[tr>0].sum()/abs(tr[tr<=0].sum())

    equity = (1+tr).cumprod()

    dd = equity/equity.cummax()-1

    sharpe = np.sqrt(len(tr))*tr.mean()/tr.std()

    print()
    print("SL =",round(sl*100,1),"%")
    print("mean =",round(tr.mean(),6))
    print("hit =",round((tr>0).mean(),3))
    print("pf =",round(pf,3))
    print("sharpe =",round(sharpe,3))
    print("maxdd =",round(dd.min(),3))
    print("equity =",round(equity.iloc[-1],3))

Stop Loss Sweep

SL = 1.0 %
mean = -0.000277
hit = 0.427
pf = 0.948
sharpe = -1.112
maxdd = -0.739
equity = 0.356

SL = 1.5 %
mean = -0.00011
hit = 0.493
pf = 0.981
sharpe = -0.393
maxdd = -0.623
equity = 0.535

SL = 2.0 %
mean = 4.9e-05
hit = 0.519
pf = 1.008
sharpe = 0.163
maxdd = -0.531
equity = 0.797

SL = 2.5 %
mean = 0.000128
hit = 0.534
pf = 1.021
sharpe = 0.404
maxdd = -0.511
equity = 0.964

SL = 3.0 %
mean = 6.7e-05
hit = 0.538
pf = 1.011
sharpe = 0.206
maxdd = -0.569
equity = 0.789

SL = 4.0 %
mean = 0.000197
hit = 0.545
pf = 1.031
sharpe = 0.581
maxdd = -0.608
equity = 1.102


In [30]:
# ============================================================
# Phase 3.7d
# Time Exit Sweep (Real Backtest)
# ============================================================

work_df = df.copy()

# ---------- build H04c signal ----------
work_df['ret'] = work_df['close'].pct_change()

work_df['sign'] = np.where(work_df['ret'] > 0, 'U', 'D')

work_df['pattern'] = (
    work_df['sign'].shift(2)
    + work_df['sign'].shift(1)
    + work_df['sign']
)

work_df['ddd_strength'] = (
    work_df['ret']
    + work_df['ret'].shift(1)
    + work_df['ret'].shift(2)
)

threshold = (
    work_df.loc[
        work_df['pattern']=='DDD',
        'ddd_strength'
    ]
    .quantile(0.10)
)

work_df['signal'] = (
    (work_df['pattern']=='DDD')
    &
    (work_df['ddd_strength'] <= threshold)
)

# ---------- sweep ----------
horizons = [6,12,24,36,48,72]

print("="*80)
print("Time Exit Sweep")
print("="*80)

for H in horizons:

    rets = []

    signal_idx = work_df.index[work_df['signal']]

    last_exit_loc = -1

    for idx in signal_idx:

        entry_loc = work_df.index.get_loc(idx)

        if entry_loc <= last_exit_loc:
            continue

        exit_loc = entry_loc + H

        if exit_loc >= len(work_df):
            break

        entry_price = work_df.iloc[entry_loc]['close']
        exit_price = work_df.iloc[exit_loc]['close']

        ret = exit_price / entry_price - 1 - 0.0004

        rets.append(ret)

        last_exit_loc = exit_loc

    tr = pd.Series(rets)

    equity = (1 + tr).cumprod()

    dd = equity / equity.cummax() - 1

    pf = tr[tr > 0].sum() / abs(tr[tr <= 0].sum())

    sharpe = np.sqrt(len(tr)) * tr.mean() / tr.std()

    print()
    print("H =", H)
    print("n =", len(tr))
    print("mean =", round(tr.mean(),6))
    print("hit =", round((tr>0).mean(),3))
    print("pf =", round(pf,3))
    print("sharpe =", round(sharpe,3))
    print("maxdd =", round(dd.min(),3))
    print("equity =", round(equity.iloc[-1],3))

Time Exit Sweep

H = 6
n = 4380
mean = -4e-06
hit = 0.536
pf = 0.999
sharpe = -0.024
maxdd = -0.647
equity = 0.763

H = 12
n = 3834
mean = 0.000186
hit = 0.546
pf = 1.044
sharpe = 0.83
maxdd = -0.59
equity = 1.405

H = 24
n = 3212
mean = 0.000351
hit = 0.537
pf = 1.066
sharpe = 1.186
maxdd = -0.507
equity = 1.955

H = 36
n = 2840
mean = 0.000579
hit = 0.549
pf = 1.097
sharpe = 1.627
maxdd = -0.513
equity = 3.067

H = 48
n = 2588
mean = 0.000685
hit = 0.538
pf = 1.101
sharpe = 1.651
maxdd = -0.621
equity = 3.294

H = 72
n = 2200
mean = 0.000562
hit = 0.546
pf = 1.072
sharpe = 1.12
maxdd = -0.543
equity = 1.863


In [31]:
for k, v in globals().copy().items():
    if isinstance(v, pd.DataFrame):
        print(f"{k:30s} {v.shape}")

df                             (631296, 5)
_4                             (5, 5)
work_df                        (631296, 10)
subset                         (901, 11)
up_subset                      (14737, 14)
down_subset                    (16821, 14)
results_df                     (8, 5)
s                              (12441, 13)
ddd_subset                     (70691, 9)
ddd                            (70691, 9)
train                          (420768, 11)
test                           (210528, 11)
train_ddd                      (46304, 9)
d                              (210528, 11)
trade_df                       (2840, 13)
wins                           (1560, 13)
losses                         (1280, 13)
hv_trades                      (1851, 10)
path                           (37, 5)


In [32]:
from pathlib import Path

out_dir = root / "data" / "backtests"
out_dir.mkdir(parents=True, exist_ok=True)

trade_df.to_parquet(
    out_dir / "h04c_trades.parquet",
    index=False
)

print("saved:", out_dir / "h04c_trades.parquet")
print("rows :", len(trade_df))

saved: d:\Projects\btc-research-lab\data\backtests\h04c_trades.parquet
rows : 2840


In [35]:
for k, v in globals().copy().items():
    if isinstance(v, pd.DataFrame):
        print(k, v.shape)

df (631296, 5)
_4 (5, 5)
work_df (631296, 10)
subset (901, 11)
up_subset (14737, 14)
down_subset (16821, 14)
results_df (8, 5)
s (12441, 13)
ddd_subset (70691, 9)
ddd (70691, 9)
train (420768, 11)
test (210528, 11)
train_ddd (46304, 9)
d (210528, 11)
trade_df (2840, 13)
wins (1560, 13)
losses (1280, 13)
hv_trades (1851, 10)
path (37, 5)


In [36]:
from pathlib import Path

out_dir = root / "data" / "backtests"
out_dir.mkdir(parents=True, exist_ok=True)

trade_df.to_parquet(
    out_dir / "h04c_trades.parquet",
    index=False
)

print("saved")
print(out_dir / "h04c_trades.parquet")
print("rows:", len(trade_df))

saved
d:\Projects\btc-research-lab\data\backtests\h04c_trades.parquet
rows: 2840


In [37]:
print(trade_df.columns.tolist())

['entry_time', 'exit_time', 'entry_price', 'exit_price', 'gross_ret', 'net_ret', 'duration', 'year', 'equity', 'vol_regime', 'bull', 'mae', 'mfe']


In [39]:
import os

print(os.getpid())

16128


In [40]:
from pathlib import Path

out_dir = root / "data" / "backtests"
out_dir.mkdir(parents=True, exist_ok=True)

trade_df.to_parquet(
    out_dir / "h04c_trades.parquet",
    index=False
)

print("saved")

saved
